# 🎙️ ProXMax — আপনার নিজের কণ্ঠে English → Bangla ডাবিং

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/redwan-primeAmax/ProXMax/blob/main/notebook/ProXMax_Colab.ipynb)

> 📌 **ধাপ ১–১০ উপর থেকে নিচে রান করুন।** আগে `Runtime → Change runtime type → T4 GPU`
> সেট করুন; ধাপ ৫-এ Hugging Face token আর ধাপ ৬-এ Gemini API key লাগবে।

এই নোটবুকটি Google Drive-এর `Video/input` ফোল্ডারের **সব English ভিডিও** একে একে বাংলায় ডাবিং করে `Video/output`-এ রাখে।

**এই সংস্করণে নতুন কী (Phase 3):**

- 🎤 **ভয়েস ক্লোনিং (IndicF5)** — আপনার নিজের কণ্ঠে ডাবিং হবে, রোবটিক ভয়েস নয়।
- ✂️ **নীরবতা-সচেতন chunking** — Whisper আর কখনো শব্দের মাঝখানে কাটে না (`XYZ` → `XY`+`Z` সমস্যা শেষ)।
- 🔁 **Rolling prompt + seam repair** — বাক্যের ধারাবাহিকতা ও punctuation ঠিক থাকে।
- ⏱️ **Duration-aware অনুবাদ (Gemini)** — প্রতিটি লাইনকে তার সময়ের বাজেট জানানো হয়।
- 🚫 **কোনো overlap নেই** — দুই কণ্ঠ কখনো একসাথে কথা বলে না (hard guarantee)।
- ⚡ **২–৩ মিনিটে ১০ মিনিটের ভিডিও** — অনুবাদ ও TTS একসাথে চলে, GPU বাজেট governor থাকে।

### শুরুর আগে তিনটি জিনিস দরকার

| দরকার | কোথায় |
|---|---|
| ভিডিও (mp4/mkv/mov/avi) | `MyDrive/Video/input/` |
| **আপনার কণ্ঠের ১০–২০ সেকেন্ডের নমুনা** (wav/mp3/m4a) | **একই `Video/input/` ফোল্ডারে** |
| Gemini API key + Hugging Face token | ধাপ ৫ ও ৬-এ বসাবেন |

> কণ্ঠের নমুনায় শান্ত ঘরে বলা স্পষ্ট বাংলা কথা থাকুক। নমুনা কত লম্বা তা নিয়ে চিন্তা নেই — কোড নিজেই সবচেয়ে ভালো অংশটি বেছে নেবে (বিরতি/নীরবতা ছেঁটে দেবে)।

## 🖥️ ধাপ ১: GPU যাচাই

আউটপুটে **Tesla T4** দেখা উচিত। GPU ছাড়া ভয়েস ক্লোনিং কার্যত চলে না।

`Runtime → Change runtime type → T4 GPU` সেট করা আছে কি না দেখে নিন।

In [ ]:
!nvidia-smi

## 📂 ধাপ ২: Google Drive mount

Drive-এ থাকা ভিডিও ও কণ্ঠের নমুনা পড়তে এবং ফলাফল সেভ করতে Drive যুক্ত করা হচ্ছে।

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

INPUT_DIR = '/content/drive/MyDrive/Video/input'
OUTPUT_DIR = '/content/drive/MyDrive/Video/output'

import os
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Input :', INPUT_DIR)
print('Output:', OUTPUT_DIR)

## ⬇️ ধাপ ৩: রিপোজিটরি clone করা

আগে থেকে clone করা থাকলে সেটি আপডেট (pull) করা হবে — নতুন কোড নিজে থেকেই চলে আসবে।

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/redwan-primeAmax/ProXMax.git'
REPO_DIR = '/content/ProXMax'

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('আগের clone পাওয়া গেছে — আপডেট করা হচ্ছে…')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

print(subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'],
                     capture_output=True, text=True).stdout.strip())

## 📦 ধাপ ৪: Dependency ইনস্টল

`requirements.txt` = মূল পাইপলাইন (Whisper, Gemini, edge-tts fallback)।
`requirements-indicf5.txt` = **ভয়েস ক্লোনিং ইঞ্জিন** (transformers, vocos, x-transformers ইত্যাদি)।

> ইনস্টল ২–৪ মিনিট নিতে পারে — শুধু প্রথমবার।

In [ ]:
%cd /content/ProXMax
!pip install -q -r requirements.txt -r requirements-indicf5.txt

In [ ]:
import shutil
import subprocess

# ffmpeg না থাকলে ইনস্টল করা (Colab-এ সাধারণত আগে থেকেই থাকে)
if shutil.which('ffmpeg') is None:
    print('ffmpeg পাওয়া যায়নি — ইনস্টল করা হচ্ছে…')
    subprocess.run('apt-get -qq update && apt-get -qq install -y ffmpeg', shell=True, check=True)

print('ffmpeg:', shutil.which('ffmpeg'))
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 🔑 ধাপ ৫: Hugging Face লগইন (gated মডেল — অবশ্যই একবার)

IndicF5 মডেলটি **gated**: ব্যবহারের আগে অনুমতি নিতে হয়।

1. এই লিঙ্কে যান: **https://huggingface.co/ai4bharat/IndicF5**
2. লগইন করে **"Agree and access repository"** চাপুন (একবারই)।
3. আপনার Hugging Face token নিন: https://huggingface.co/settings/tokens (`read` permission যথেষ্ট)।
4. Colab-এর বাঁ দিকের 🔑 **Secrets**-এ `HF_TOKEN` নামে token রেখে দিন (সুপারিশ) — নাহলে নিচের সেলে লিখতে হবে।

> token কোথাও পাবলিকভাবে শেয়ার করবেন না।

In [ ]:
import os

HF_TOKEN = ''
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass('Hugging Face token (hf_...): ').strip()

from huggingface_hub import login, model_info

login(token=HF_TOKEN, add_to_git_credential=False)
os.environ['HF_TOKEN'] = HF_TOKEN

try:
    info = model_info('ai4bharat/IndicF5')
    print('✅ মডেল access ঠিক আছে:', info.id)
except Exception as exc:
    print('❌ IndicF5 মডেলে access নেই — উপরের ধাপ ৫-এর ১ ও ২ নম্বর লিঙ্কে অনুমতি নিন।')
    print('   বিস্তারিত:', exc)

## 🔑 ধাপ ৬: Gemini API key

অনুবাদ হয় **Google Gemini API** দিয়ে (ডিফল্ট মডেল `gemini-3.5-flash-light`)।

- key নিন: https://aistudio.google.com/app/apikey
- Colab Secrets-এ `GEMINI_API_KEY` নামে রাখুন (সুপারিশ), নাহলে নিচের সেলে লিখুন।

In [ ]:
import os

GEMINI_API_KEY = ''
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY') or ''
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')

if not GEMINI_API_KEY:
    from getpass import getpass
    GEMINI_API_KEY = getpass('Gemini API key: ').strip()

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
print('Gemini key সেট হয়েছে:', 'হ্যাঁ' if GEMINI_API_KEY else 'না')

## ⚙️ ধাপ ৭: দ্রুত সেটিংস (ঐচ্ছিক)

না বদলালেও চলবে — ডিফল্ট মানই সর্বোচ্চ গুণমানের। শুধু দরকার হলে বদলান।

| সেটিং | ডিফল্ট | কখন বদলাবেন |
|---|---|---|
| `WHISPER_MODEL` | `large-v3` | খুব ধীর লাগলে `medium` |
| `TTS_ENGINE` | `indicf5` | শুধু কণ্ঠ ক্লোনিং ছাড়া ডাবিং চাইলে `edge` |
| `MAX_TEMPO` | `1.25` | এর বেশি দিলে কণ্ঠ রোবটিক শোনাবে (সুপারিশ: বদলাবেন না) |

In [ ]:
WHISPER_MODEL = 'large-v3'   # 'large-v3' (best) | 'medium' (faster)
TTS_ENGINE = 'indicf5'       # 'indicf5' (আপনার কণ্ঠ) | 'edge' (ক্লোনিং ছাড়া)
MAX_TEMPO = 1.25             # 1.25-এর বেশি নয় — নইলে কণ্ঠ বিকৃত হয়

args = []
if WHISPER_MODEL != 'large-v3':
    args += ['--model', WHISPER_MODEL]
if TTS_ENGINE != 'indicf5':
    args += ['--tts-engine', TTS_ENGINE]
print('অতিরিক্ত আর্গুমেন্ট:', args or '(কিছু নেই — ডিফল্ট)')

## ▶️ ধাপ ৮: ডাবিং শুরু

`Video/input`-এর সব ভিডিও একটার পর একটা প্রসেস হবে।

**প্রথম রানে** IndicF5 মডেল (~১.৪ GB) আর Whisper মডেল (~৩ GB) ডাউনলোড হবে — একটু সময় লাগবে। পরেরবার ক্যাশ থেকে লোড হবে।

লগে খেয়াল করুন: `ধাপ ১/৭ … ধাপ ৭/৭` শেষ হলে ভিডিও তৈরি। কোনো ভিডিও ব্যর্থ হলেও বাকিগুলো চলবে, শেষে সারসংক্ষেপ দেখাবে।

In [ ]:
%cd /content/ProXMax
!python main.py

## 🎉 ধাপ ৯: ফলাফল দেখা

প্রতিটি ভিডিওর নামের শেষে `_bn.mp4` — সেটাই ডাবিং করা ভিডিও। পাশে `_en.srt` (মূল) ও `_bn.srt` (বাংলা) সাবটাইটেল।

In [ ]:
import os
import subprocess

files = sorted(os.listdir(OUTPUT_DIR)) if os.path.isdir(OUTPUT_DIR) else []
videos = [f for f in files if f.lower().endswith('.mp4')]

print(f'মোট ফাইল: {len(files)} | ডাবিং করা ভিডিও: {len(videos)}\n')
for name in files:
    path = os.path.join(OUTPUT_DIR, name)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f'  {name:45s} {size_mb:7.1f} MB')

if videos:
    print('\nDrive থেকে সরাসরি দেখা/ডাউনলোড করা যাবে:', os.path.join(OUTPUT_DIR, videos[0]))

In [ ]:
# প্রথম ডাবিং করা ভিডিওটি ব্রাউজারে চালিয়ে শোনা (Drive থেকে)
from IPython.display import HTML, display
import os

videos = [f for f in sorted(os.listdir(OUTPUT_DIR)) if f.lower().endswith('_bn.mp4')]
if videos:
    path = os.path.join(OUTPUT_DIR, videos[0])
    display(HTML(f'<video controls width="640" src="/content/drive/MyDrive/Video/output/{videos[0]}"></video>'))
    print('ভিডিও:', path)
else:
    print('কোনো ডাবিং করা ভিডিও পাওয়া যায়নি — ধাপ ৮-এর লগ দেখুন।')

## 🧹 ধাপ ১০: GPU মেমরি ছেড়ে দেওয়া

ডাবিং শেষে প্রোগ্রাম নিজেই মডেল আনলোড করে GPU ছেড়ে দেয়। তবু নিশ্চিত করতে এই সেলটি রান করুন (Colab সেশন চালু রাখলেও VRAM খালি থাকবে)।

In [ ]:
import gc

for name in ('pipeline', 'model', 'tts_engine'):
    globals()[name] = None
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        used = torch.cuda.memory_allocated() / (1024 ** 3)
        print(f'GPU মেমরি এখন ব্যবহৃত: {used:.2f} GB (খালি করা হয়েছে)')
    else:
        print('GPU পাওয়া যায়নি — CPU মোড।')
except Exception as exc:
    print('torch ছাড়াই পরিষ্কার করা হলো:', exc)

## 🩺 সমস্যা হলে

| লক্ষণ | কারণ ও সমাধান |
|---|---|
| `IndicF5 মডেলটি gated` | ধাপ ৫-এর ১ ও ২ নম্বর লিঙ্কে অনুমতি নিন, token ঠিক আছে কি না দেখুন |
| `ইনপুট ফোল্ডারে কোনো ভয়েস নমুনা পাওয়া যায়নি` | `Video/input`-এ আপনার ১০–২০s বাংলা কথার audio ফাইল রাখুন |
| `CUDA out of memory` | Runtime → Restart, ধাপ ৮ আবার চালান; নোটবুক নিজেই TTS-এর আগে Whisper আনলোড করে |
| `Gemini API error (429)` | quota শেষ — কিছুক্ষণ অপেক্ষা করুন; কোড নিজেই ৭ বার retry করে |
| `ffmpeg পাওয়া যায়নি` | ধাপ ৪-এর ffmpeg সেলটি রান করুন |
| ডাবিং ধীর লাগছে | `WHISPER_MODEL='medium'` করুন; এবং `config.yaml`-এর `tts.time_budget_factor` কমানো যাবে |

বিস্তারিত: রিপোর `README.md`।